# Анализ данных домена Marketplace за последние 30 дней

В домене Marketplace собраны логи пользователей на маркетплейсе: просмотры, клики, лайки и переходы к покупке. События снабжены контекстами о том, где они происходили: в поиске, каталоге, на главной странице, в рекомендациях (u2i, i2i). Для товаров доступен каталог с брендом, embedding и категориями, но не для всех товаров категории заполнены. https://habr.com/ru/companies/tbank/articles/950696/

In [1]:
import os
import pandas as pd
import numpy as np
import glob

In [2]:
events_dir = "../data/raw/T-bank dataset full/T-ECD-small/dataset/small/marketplace/events"

In [3]:
# Получаем список всех файлов .pq
event_files = [f for f in os.listdir(events_dir) if f.endswith('.pq')]

In [4]:
# Отсортируем по номеру дня
event_files_sorted = sorted(event_files, reverse=True)

In [5]:
# Возьмем последние 30 файлов
last_30_files = event_files_sorted[:30]

In [6]:
# Считываем и объединяем данные
dfs = []
for fname in last_30_files:
    filepath = os.path.join(events_dir, fname)
    df = pd.read_parquet(filepath)
    dfs.append(df)

In [7]:
# Объединенный DataFrame за последние 30 дней
recent_events = pd.concat(dfs, ignore_index=True)

In [8]:
users_df = pd.read_parquet('../data/raw/T-bank dataset full/T-ECD-small/dataset/small/users.pq')
items_df = pd.read_parquet('../data/raw/T-bank dataset full/T-ECD-small/dataset/small/marketplace/items.pq')

In [9]:
brands_df = pd.read_parquet('../data/raw/T-bank dataset full/T-ECD-small/dataset/small/brands.pq', engine='fastparquet')

# Исследуем таблицу recent_events

In [10]:
recent_events.head()

,timestamp,user_id,item_id,subdomain,action_type,os
0,113011200183108,73774100,nfmcg_18200186,other,view,android
1,113011200333142,38161973,nfmcg_14129567,other,view,android
2,113011200570986,52431976,nfmcg_4396804,search,view,android
3,113011200656523,75059116,nfmcg_27122678,i2i,view,other
4,113011200923703,10628666,nfmcg_6136132,other,view,android


In [11]:
recent_events.shape

(17790404, 6)

In [12]:
print(f'Уникальных пользователей: {recent_events.user_id.nunique()}')
print(f'Уникальных айтемов: {recent_events.item_id.nunique()}')

Уникальных пользователей: 660068
Уникальных айтемов: 844803


In [13]:
recent_events.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17790404 entries, 0 to 17790403
Data columns (total 6 columns):
 #   Column       Dtype 
---  ------       ----- 
 0   timestamp    int64 
 1   user_id      uint64
 2   item_id      object
 3   subdomain    object
 4   action_type  object
 5   os           object
dtypes: int64(1), object(4), uint64(1)
memory usage: 814.4+ MB


#### Распределение операционных систем

In [14]:
recent_events['os'].unique()

array(['android', 'other', 'ios'], dtype=object)

In [15]:
recent_events.groupby('os', dropna=False)['user_id'].count().sort_values(ascending=False)

os
android    12466722
other       3120929
ios         2202753
Name: user_id, dtype: int64

Из таблицы видно, что чаще всего используемая платформа - android

In [16]:
recent_events['action_type'].unique()

array(['view', 'click', 'clickout', 'like'], dtype=object)

#### Распределение типов действий

In [17]:
recent_events.groupby('action_type', dropna=False)['user_id'].count().sort_values(ascending=False)

action_type
view        17279844
click         431008
clickout       73489
like            6063
Name: user_id, dtype: int64

Самое частое действие - просмотр.

#### Распределение событий по контекстам, где они происходили (рекомендации, каталог, поиск, похожие товары)

In [18]:
recent_events['subdomain'].unique()

array(['other', 'search', 'i2i', 'u2i', 'catalog', None], dtype=object)

In [19]:
recent_events.groupby('subdomain', dropna=False)['user_id'].count().sort_values(ascending=False)

subdomain
u2i        8184685
i2i        2723162
catalog    2426842
other      2295871
search     2153781
NaN           6063
Name: user_id, dtype: int64

Из таблицы видно, что наибольшее количество взаимодействий пользователей с товаром произошло когда товар был показан в блоке персональных рекомендаций, сформированных специально для этого пользователя.

### Работа со временем. Предположение - время задано в микросекундах со сдвигом

In [20]:
recent_events['timestamp'].min(), recent_events['timestamp'].max()

(np.int64(110505600037405), np.int64(113097598175560))

In [21]:
# конвертируем в datetime (получим примерно 1973 год)
recent_events['datetime_wrong_year'] = pd.to_datetime(recent_events['timestamp'], unit='us')

In [22]:
# желаемая и фактическая даты начала
# .normalize() убирает время, оставляя только дату
desired_start_date = pd.to_datetime('2025-01-01')
actual_start_date = recent_events['datetime_wrong_year'].min().normalize() 

In [23]:
# сдвиг
time_offset = desired_start_date - actual_start_date

In [24]:
# применяем сдвиг ко всем датам
recent_events['datetime_corrected'] = recent_events['datetime_wrong_year'] + time_offset

In [25]:
print(recent_events[['timestamp', 'datetime_wrong_year', 'datetime_corrected']].sort_values(by='datetime_corrected').head())

                timestamp        datetime_wrong_year  \
16989295  110505600037405 1973-07-03 00:00:00.037405   
16989296  110505600037405 1973-07-03 00:00:00.037405   
16989297  110505600219769 1973-07-03 00:00:00.219769   
16989298  110505600655919 1973-07-03 00:00:00.655919   
16989299  110505601306735 1973-07-03 00:00:01.306735   

                 datetime_corrected  
16989295 2025-01-01 00:00:00.037405  
16989296 2025-01-01 00:00:00.037405  
16989297 2025-01-01 00:00:00.219769  
16989298 2025-01-01 00:00:00.655919  
16989299 2025-01-01 00:00:01.306735  


In [27]:
print(recent_events[['timestamp', 'datetime_wrong_year', 'datetime_corrected']].sort_values(by='datetime_corrected').tail())

              timestamp        datetime_wrong_year         datetime_corrected
533349  113097596634823 1973-08-01 23:59:56.634823 2025-01-30 23:59:56.634823
533350  113097596970248 1973-08-01 23:59:56.970248 2025-01-30 23:59:56.970248
533351  113097597225981 1973-08-01 23:59:57.225981 2025-01-30 23:59:57.225981
533352  113097598066187 1973-08-01 23:59:58.066187 2025-01-30 23:59:58.066187
533353  113097598175560 1973-08-01 23:59:58.175560 2025-01-30 23:59:58.175560


In [28]:
print(recent_events['datetime_corrected'].max() - recent_events['datetime_corrected'].min())

29 days 23:59:58.138155


# Исследуем данные из таблицы users_df

In [20]:
users_df.head()

,user_id,socdem_cluster,region
0,77309558,21.0,2.0
1,72517894,10.0,90.0
2,86699708,9.0,9.0
3,54241043,17.0,58.0
4,23591057,17.0,4.0


In [21]:
users_df.shape

(3500000, 3)

In [22]:
users_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3500000 entries, 0 to 3499999
Data columns (total 3 columns):
 #   Column          Dtype  
---  ------          -----  
 0   user_id         uint64 
 1   socdem_cluster  float64
 2   region          float64
dtypes: float64(2), uint64(1)
memory usage: 80.1 MB


In [23]:
users_df['socdem_cluster'].nunique()

21

In [24]:
users_df['region'].nunique()

90

#### Распределние по социо-демографическим кластерам (первые пять наиболее заполненнных)

In [25]:
users_df.groupby('socdem_cluster', dropna=False)['user_id'].count().sort_values(ascending=False).head()

socdem_cluster
17.0    473629
20.0    420944
12.0    416287
9.0     340144
21.0    331140
Name: user_id, dtype: int64

#### Распределение по регионам

In [26]:
users_df.groupby('region', dropna=False)['user_id'].count().sort_values(ascending=False).head()

region
2.0     551358
60.0    249920
90.0    181623
18.0    137250
24.0    114992
Name: user_id, dtype: int64

# Исследуем данные из таблицы items_df

In [27]:
items_df.head()

,item_id,brand_id,category,subcategory,price,embedding
0,nfmcg_10000001,137356,None,None,6.063640,"[-0.07085289, 0.032460444, 0.009126052, 0.0377..."
1,nfmcg_10000012,53389,None,None,0.960436,"[-0.09109873, 0.043168057, -0.027675372, 0.031..."
2,nfmcg_1000002,34153,"Fashion Accessories, Tech Add-ons, and Style E...","Hats, Scarves, and Shawls",-1.793113,"[-0.05653296, 0.08287799, -0.06381639, 0.09553..."
3,nfmcg_10000034,39543,None,None,3.416755,"[-0.11258836, 0.04333279, 0.029613094, -0.0040..."
4,nfmcg_10000039,82320,"Fashion Accessories, Tech Add-ons, and Style E...",Jewelry and Costume Jewelry,4.293433,"[-0.15720145, 0.02623378, 0.005749651, 0.06972..."


In [28]:
items_df.shape

(2325409, 6)

In [29]:
items_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2325409 entries, 0 to 2325408
Data columns (total 6 columns):
 #   Column       Dtype  
---  ------       -----  
 0   item_id      object 
 1   brand_id     uint64 
 2   category     object 
 3   subcategory  object 
 4   price        float64
 5   embedding    object 
dtypes: float64(1), object(4), uint64(1)
memory usage: 106.4+ MB


In [30]:
items_df['category'].unique()

array([None, 'Fashion Accessories, Tech Add-ons, and Style Enhancements',
       'Miscellaneous Goods (Uncategorized)',
       'Cosmetics, Personal Care, and Health Maintenance Products',
       'Footwear of All Types',
       'Home/Office Furniture and Interior Decor',
       'Sports Equipment, Gear, and Outdoor Activity Accessories',
       'Electronic Devices and Gadgets',
       'Construction and Renovation Materials, Tools, and Equipment',
       'Home Improvement and Countryside Retreat Essentials',
       "Children's Products and Childcare Items",
       'Outerwear, Casual Apparel, and Specialized Workwear',
       'Household Electrical Appliances', 'Foodstuffs and Beverages',
       'Office Supplies and Educational Materials',
       'Pharmaceuticals and Medical Supplies',
       'Hobby, Creative, and Leisure Products',
       'Cleaning Supplies and Everyday Household Items',
       'Pet Supplies: Food, Accessories, and Grooming Products',
       'Automotive Accessories, Spare 

In [31]:
items_df.groupby('category', dropna=False)['item_id'].count().sort_values(ascending=False)

category
NaN                                                            966395
Miscellaneous Goods (Uncategorized)                            266146
Cosmetics, Personal Care, and Health Maintenance Products      188412
Fashion Accessories, Tech Add-ons, and Style Enhancements      167811
Outerwear, Casual Apparel, and Specialized Workwear            153223
Footwear of All Types                                          103680
Electronic Devices and Gadgets                                  84282
Home Improvement and Countryside Retreat Essentials             83707
Construction and Renovation Materials, Tools, and Equipment     83584
Home/Office Furniture and Interior Decor                        64919
Household Electrical Appliances                                 61762
Sports Equipment, Gear, and Outdoor Activity Accessories        36214
Hobby, Creative, and Leisure Products                           13292
Children's Products and Childcare Items                         13161
Automotive 

In [32]:
items_df['subcategory'].nunique()

192

192 уникальные подкатегории

In [33]:
items_df.groupby('subcategory', dropna=False)['item_id'].count().sort_values(ascending=False)

subcategory
NaN                                        1233023
Jewelry and Costume Jewelry                  99751
Women's Clothing                             82778
Women's Footwear                             62212
Men's Clothing                               46759
                                            ...   
Veterinary Medications and Products              7
Educational and Instructional Materials          7
Meat and Sausage Products                        6
Farm Produce                                     6
Horse and Equestrian Supplies                    1
Name: item_id, Length: 193, dtype: int64

# Посмотрим на данные из таблицы brands_df

In [34]:
brands_df.head()

,brand_id,embedding
0,4,None
1,34,None
2,45,None
3,46,None
4,51,None


In [35]:
brands_df.shape

(24513, 2)

In [36]:
brands_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24513 entries, 0 to 24512
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   brand_id   24513 non-null  uint64
 1   embedding  6559 non-null   object
dtypes: object(1), uint64(1)
memory usage: 383.1+ KB


# Анализ пропусков

Пропущенные значения в датасете users_df

In [37]:
users_df.isna().sum()

user_id               0
socdem_cluster     5153
region            58917
dtype: int64

In [38]:
def missing_value_rate(df, column):
  print(f'Доля пропущенных значений в колонке {column}:  {(df[column].isna().sum() / users_df.shape[0]):.4f}')

In [39]:
for column in users_df.columns:
  missing_value_rate(users_df, column)

Доля пропущенных значений в колонке user_id:  0.0000
Доля пропущенных значений в колонке socdem_cluster:  0.0015
Доля пропущенных значений в колонке region:  0.0168


Возможные причины возникновения пропусков:
Необязательные поля при регистрации
Система не смогла определить местоположение пользователя, возможно пользователь не дал разрешение на определение местоположения
Технические ошибки при сборе данных

In [40]:
print(users_df[(users_df['socdem_cluster'].isna()) & (users_df['region'].isna())].shape[0])
print(users_df[(users_df['socdem_cluster'].isna()) & (users_df['region'].isna())].shape[0] / users_df.shape[0])

4667
0.0013334285714285713


Итого у нас есть 4667 записей пользователей, где пропуски в обеих колонках socdem_cluster и region. Это около 1,3%.

In [41]:
# проверим дубликаты
users_df.duplicated().sum()

np.int64(0)

In [42]:
print(f'Минимальное значение в колонках socdem_cluster и region соответвенно: {users_df['socdem_cluster'].min()} и {users_df['region'].min()}')

Минимальное значение в колонках socdem_cluster и region соответвенно: 0.0 и 0.0


Стратегия работы с пропусками
Процент пропусков очень мал. Технически можно удалить строки с пропущенными значениями в таблице users_df. Но тогда бы должны будем удалить этих пользователей из таблицы events_df, которую мы будем использовать для посторения матрицы взаимодействий и уже на этой основе строить прогноз. В таблице events_df нет пропусков.

Колонки socdem_cluster и region это категориальные данные, закодированные числами. Нам нужно заполнить пропущенные значения специальным зарезервированным числом, которое не встречается среди реальных кодов регионов и кластеров. Таким образом, мы сохраним всю историю взаимодействий c этими айтемами в events_df. Можно использовать число -1, так как минимальное значение в указанных колонках равно ноль.

# Пропущенные значения в items_df

In [43]:
items_df.isna().sum()

item_id              0
brand_id             0
category        966395
subcategory    1233023
price             2882
embedding           73
dtype: int64

In [44]:
for column in items_df.columns:
  missing_value_rate(items_df, column)

Доля пропущенных значений в колонке item_id:  0.0000
Доля пропущенных значений в колонке brand_id:  0.0000
Доля пропущенных значений в колонке category:  0.2761
Доля пропущенных значений в колонке subcategory:  0.3523
Доля пропущенных значений в колонке price:  0.0008
Доля пропущенных значений в колонке embedding:  0.0000


### Возможные причины возникновения пропусков

Для category и subcategory - товар могли добавить в базу данных, но еще не успели присвоить ему категорию.

Некоторые товары могут не подходить ни под одну из существующих категорий, и поле остается пустым.

Поставшики прислали неполную информацию.

Для price - это могут быть товары не для продажи, а, например, промо-материалы.

Некоторые товары могут не иметь публичной цены, а она узнается по запросу.

Часть комплекта: товар может продаваться только в составе набора, и у него нет индивидуальной цены.

### Стратегия работы с пропусками

* в колонках `category` и `subcategory` (категориальные данные) заменить на 'unknown'.Таким образом, мы сохраняем всю его ценную историю взаимодействий в `events_df`.
* Это лучше, чем заполнять самой частой категорией, так как это может внести смещение в данные.​
* Колонка `price`. Можно заполнить нулем, если мы можем предположить, что товар, например, акционный, бесплатный. Но это частный случай и у нас нет данных для выдвижения такого предположения. Лучше заполнить медианой. Медиана более устойчива к выбросам, поэтому она лучше отражает "типичную" цену.

# Пропущенные значения в events_df

In [45]:
recent_events.isna().sum()

timestamp         0
user_id           0
item_id           0
subdomain      6063
action_type       0
os                0
dtype: int64

In [46]:
recent_events['subdomain'].unique()

array(['other', 'search', 'i2i', 'u2i', 'catalog', None], dtype=object)

subdomain - область, на которой происходило взаимодействие (рекомендации, каталог, поиск, оформление заказа, кампания).

In [47]:
missing_value_rate(recent_events, 'subdomain')

Доля пропущенных значений в колонке subdomain:  0.0017


### Стратегия работы с пропусками

* в колонке `subdomain` заменить на 'unknown'.

# Пропущенные значения в brands_df

In [48]:
brands_df.isna().sum()

brand_id         0
embedding    17954
dtype: int64

In [49]:
print(f'Доля пропущенных значений в колонке embedding:  {(brands_df['embedding'].isna().sum() / brands_df.shape[0]):.4f}')

Доля пропущенных значений в колонке embedding:  0.7324


Более 73% брендов не имеют эмбеддинга.

* **Проблема холодного старта.** Эмбеддинги брендов обычно строятся на основе поведения пользователей (просмотры, покупки, клики). Если бренд новый или с ним было очень мало взаимодействий, у системы недостаточно данных, чтобы построить для него осмысленный эмбеддинг. Эти пропущенные эмбеддинги - это новые или непопулярные бренды.

* Эмбеддинги могут пересчитывать не в реальном времени, а, например, раз в неделю. Пропуски могут соответствовать брендам, которые были добавлены после последнего обновления и еще ждут своей очереди на обработку.

### Стратегрия работы с пропусками

* Удалять нельзя.
* На данном этапе оставить без изменения.
* На этапе моделирования заменить все NaN на специальный обучаемый неизвестный эмбеддинг, чтобы дать модели самой выучить специальный, 
единый эмбеддинг для всех "неизвестных" брендов.
* Со звездочкой. Использовать гибридный подход: Для "холодных" брендов можно пытаться сгенерировать временный эмбеддинг 
на основе их других характеристик (категория, цена, текстовое описание, если оно есть). Это называется гибридной рекомендательной 
системой, которая смешивает коллаборативную фильтрацию с подходом на основе контента

# Заполняем пропуски, чтобы потом посмотреть основные статистики

## Обработка users_df
В таблице users_df заполняем пропуски в закодированных категориальных признаках region и socdem_cluster числом -1. Мы это можем сделать, так как минимальные значения этих признаков равны нулю.

In [50]:
users_df['region'] = users_df['region'].fillna(-1)
users_df['socdem_cluster'] = users_df['socdem_cluster'].fillna(-1)

In [51]:
# Меняем тип данных на целочисленный
users_df['region'] = users_df['region'].astype(int)
users_df['socdem_cluster'] = users_df['socdem_cluster'].astype(int)

In [52]:
print('Пропуски в users_df после обработки:')
print(users_df.isnull().sum())

Пропуски в users_df после обработки:
user_id           0
socdem_cluster    0
region            0
dtype: int64


## Обработка items_df

In [53]:
# Заполняем категориальные пропуски строкой 'unknown'
items_df['category'] = items_df['category'].fillna('unknown')
items_df['subcategory'] = items_df['subcategory'].fillna('unknown')

In [54]:
# Рассчитываем медианную цену
median_price = items_df['price'].median()
print(f'Медианная цена для заполнения пропусков: {median_price}')

Медианная цена для заполнения пропусков: 0.42575823138244323


In [55]:
items_df['price'].min()

np.float64(-10.0)

В признаке price присутствуют отрицательные значения. Это не выбросы. В описании дата сета price - представляет собой денежную стоимость взаимодействия. Цены в каталогах переведены в промежуток от −10 до 10. 

In [56]:
# Заполняем пропуски в цене медианой
items_df['price'] = items_df['price'].fillna(median_price)

In [57]:
print("Пропуски в items_df после обработки:")
print(items_df.isnull().sum())

Пропуски в items_df после обработки:
item_id         0
brand_id        0
category        0
subcategory     0
price           0
embedding      73
dtype: int64


## Обработка brands_df

Пропуски в embedding не трогаем. Они важны как сигнал о "холодных" брендах.

# Основные статистики

### Анализ таблицы users_df

Признаки 'socdem_cluster', 'region' это категориальные, закодированные числами, метод describe тут не поможет.

In [58]:
print(f'Уникальных пользователей: {users_df['user_id'].nunique()}')
print(f'Уникальных регионов: {users_df['region'].nunique()}')
print(f'Уникальных социо-демографических кластеров: {users_df['socdem_cluster'].nunique()}')

Уникальных пользователей: 3500000
Уникальных регионов: 91
Уникальных социо-демографических кластеров: 22


In [59]:
# Посмотрим, сколько пользователей в каждом регионе
users_df['region'].value_counts()

region
2     551358
60    249920
90    181623
18    137250
24    114992
       ...  
65      1687
30      1125
69       915
39       836
20       338
Name: count, Length: 91, dtype: int64

In [60]:
# Посмотрим, сколько пользователей в каждом кластере
users_df['socdem_cluster'].value_counts()

socdem_cluster
 17    473629
 20    420944
 12    416287
 9     340144
 21    331140
 10    266045
 19    261186
 4     254800
 0     205299
 5     148037
 7     140750
 18    105353
 13     40572
 6      36130
 1      18629
 2      12134
 16      9859
 8       5825
-1       5153
 3       3900
 11      2637
 15      1547
Name: count, dtype: int64

In [61]:
# Преобразуем столбцы в категориальный тип
users_df['socdem_cluster'] = users_df['socdem_cluster'].astype('category')
users_df['region'] = users_df['region'].astype('category')

In [62]:
# И применим describe
print(users_df[['socdem_cluster', 'region']].describe())

        socdem_cluster   region
count          3500000  3500000
unique              22       91
top                 17        2
freq            473629   551358


### Анализ items_df

In [63]:
desc = items_df['price'].describe()
print(desc.apply(lambda x: f'{x:.5f}'))

count    2325409.00000
mean           0.48972
std            2.45352
min          -10.00000
25%           -1.21740
50%            0.42576
75%            2.19184
max           10.00000
Name: price, dtype: object


* Из таблицы видно, что среднее значение и медиана близки друг к другу, значит распределение относительно симметричное.
Имеет место правостороннее распределение с положительной асимметрией, где среднее значение 0.49 и медиана 0.43 смещены в положительную область, что указывает на преобладание цен выше условного нуля. Четверть значений остаются отрицательными (25-й перцентиль: -1.22), при этом основная масса данных сосредоточена в интервале от -1.22 до 2.19.
  
* Максимальное значение 10 очень сильно отличается от 75-го процентиля. Это явный признак наличия выбросов. Наличие товаров, которые значительно дороже основной массы товаров.
* Минимальное значение -10.000000 тоже сильно отличается от 25-го процентиля. Значит есть товары, которые значительно дешевле основной массы.

### Анализ events_df

In [64]:
recent_events.head()

,timestamp,user_id,item_id,subdomain,action_type,os
0,113011200183108,73774100,nfmcg_18200186,other,view,android
1,113011200333142,38161973,nfmcg_14129567,other,view,android
2,113011200570986,52431976,nfmcg_4396804,search,view,android
3,113011200656523,75059116,nfmcg_27122678,i2i,view,other
4,113011200923703,10628666,nfmcg_6136132,other,view,android


#### Определение самых популярных товаров

События типа 'view' фиксируют только факт показа товара пользователю, и многие из них могут быть случайными или малозначимыми (пользователь просто пролистал страницу, товар попал в рекомендацию, но не заинтересовал). Поэтому топ по просмотрам покажет товары, которые чаще всего показывались, но не обязательно были востребованы.

Для выявления действительно популярных и интересных товаров лучше использовать более сильные сигналы, такие как:

'click' — пользователь проявил явный интерес, кликнув на товар.

'like' — проявил позитивную реакцию, добавил в избранное.

'clickout' — намерение к покупке, переход к оформлению или внешнему сайту.

In [65]:
# Вспомним, какие есть типы взаимодействий
recent_events['action_type'].unique()

array(['view', 'click', 'clickout', 'like'], dtype=object)

In [66]:
# Берем только клики и считаем количества кликов для каждого item_id
top_popular_items = recent_events[recent_events['action_type'] == 'click']['item_id'].value_counts()

# Выводим топ-10 самых популярных товаров по кликам
print(top_popular_items.head(10))

item_id
nfmcg_20987893    10323
nfmcg_22178359     7998
nfmcg_13647154     6349
nfmcg_4104717      6310
nfmcg_14380950     6108
nfmcg_8961427      5887
nfmcg_28106008     4453
nfmcg_3698542      4292
nfmcg_22211380     4157
nfmcg_6301488      3679
Name: count, dtype: int64


In [67]:
# Берем только clickout и считаем количества clickout для каждого item_id
top_popular_items = recent_events[recent_events['action_type'] == 'clickout']['item_id'].value_counts()

# Выводим топ-10 самых популярных товаров
print(top_popular_items.head(10))

item_id
nfmcg_20987893    1490
nfmcg_4104717     1090
nfmcg_22178359     705
nfmcg_18200186     698
nfmcg_22437058     674
nfmcg_16225713     646
nfmcg_13647154     635
nfmcg_6136132      628
nfmcg_28106008     627
nfmcg_6613648      553
Name: count, dtype: int64


In [68]:
# Берем только лайки и считаем количества лайков для каждого item_id
top_popular_items = recent_events[recent_events['action_type'] == 'like']['item_id'].value_counts()

# Выводим топ-10 самых популярных товаров по лайкам
print(top_popular_items.head(10))

item_id
nfmcg_20987893    118
nfmcg_4104717      88
nfmcg_18106422     68
nfmcg_13647154     61
nfmcg_28106008     53
nfmcg_22437058     52
nfmcg_6136132      52
nfmcg_9717065      51
nfmcg_8961427      47
nfmcg_22178359     45
Name: count, dtype: int64


Создадим столбец label. Действию view в колонке label поставим в соответсвие 0, а действиям click, clickout, like поставим в соответсвие 1.

In [69]:
recent_events['label'] = recent_events['action_type'].map({'view':0, 'click':1, 'clickout':1, 'like': 1})

In [70]:
recent_events.groupby('action_type')['label'].mean()

action_type
click       1.0
clickout    1.0
like        1.0
view        0.0
Name: label, dtype: float64

In [71]:
# Популярность по количеству уникальных пользователей. 
# Именно такой способ расчета популярности часто используется в рекомендательных системах

# Считаем количество уникальных пользователей, совершивших значимые действия для каждого товара
label_1_recent_events = recent_events[recent_events['label'] == 1]
unique_user_popularity = label_1_recent_events.groupby('item_id')['user_id'].nunique().sort_values(ascending=False)

print(unique_user_popularity.head(10))

item_id
nfmcg_20987893    10604
nfmcg_22178359     7491
nfmcg_4104717      6726
nfmcg_13647154     6638
nfmcg_8961427      5753
nfmcg_14380950     5610
nfmcg_28106008     4849
nfmcg_3698542      4240
nfmcg_22211380     4230
nfmcg_6301488      3869
Name: user_id, dtype: int64


# Анализ взаимосвязи переменных

В items_df содержится больше всего прикзнаков, которые могут быть взаимосвязаны. Добавим к ним признаки из events. И все категориальные закодируем числами.

In [72]:
# Нам нужны признаки в числовом формате
# Кодирование категориальных признаков
# Метод .cat.codes присваивает уникальный числовой код каждой категории.

Важно понимать, что .cat.codes можно использовать для анализа корреляций. Но нельзя использовать для линейных моделей. Для линейных моделей лучше использовать one-hot encoding

In [73]:
recent_events.columns

Index(['timestamp', 'user_id', 'item_id', 'subdomain', 'action_type', 'os',
       'label'],
      dtype='object')

In [74]:
items_df.columns

Index(['item_id', 'brand_id', 'category', 'subcategory', 'price', 'embedding'], dtype='object')

In [75]:
# Объединяем датафреймы items_df и recent_events
item_features = items_df[['item_id', 'price', 'brand_id', 'category', 'subcategory']]
merged_df = pd.merge(recent_events, item_features, on='item_id', how='left')

In [76]:
merged_df.columns

Index(['timestamp', 'user_id', 'item_id', 'subdomain', 'action_type', 'os',
       'label', 'price', 'brand_id', 'category', 'subcategory'],
      dtype='object')

In [77]:
# Список всех категориальных колонок, которые мы хотим проанализировать
categorical_features = ['brand_id', 'category', 'subcategory', 'subdomain', 'os']

In [78]:
analysis_df = merged_df.copy()

In [79]:
#Кодируем категории
for feature in categorical_features:
    analysis_df[f'{feature}_encoded'] = analysis_df[feature].astype('category').cat.codes

In [80]:
# Выбираем столбцы для корреляционной матрицы: цена + все закодированные признаки
features_for_heatmap = ['price'] + [f'{feature}_encoded' for feature in categorical_features]

In [81]:
correlation_matrix_full = analysis_df[features_for_heatmap].corr()
correlation_matrix_full

,price,brand_id_encoded,category_encoded,subcategory_encoded,subdomain_encoded,os_encoded
price,1.000000,-0.308250,-0.207657,-0.156252,0.069012,-0.126071
brand_id_encoded,-0.308250,1.000000,-0.030309,-0.172410,0.020410,-0.021517
category_encoded,-0.207657,-0.030309,1.000000,0.717004,-0.068744,0.058248
subcategory_encoded,-0.156252,-0.172410,0.717004,1.000000,-0.073478,0.065338
subdomain_encoded,0.069012,0.020410,-0.068744,-0.073478,1.000000,-0.218080
os_encoded,-0.126071,-0.021517,0.058248,0.065338,-0.218080,1.000000


* Большинство взаимосвязей довольно слабые, что снижает риски мультиколлинеарности при построении моделей
* Категория и подкатегория имеют очень высокую положительную корреляцию (0.72) - это логично, так как подкатегории вложены в категории
* Бренд является наиболее значимым фактором для цены среди рассмотренных признаков
* Операционная система и поддомен (место взаимодействия) имеют заметную отрицательную корреляцию (-0.22). Пользователи на разных операционных системах имеют разные паттерны взаимодействия с платформой. Например, может быть так, что пользователи чаще используют поиск на одной ос, а на другой - рекомендации.

# Целевая переменная

На основе семинара-лекции: https://vkvideo.ru/video-123851409_456239391?list=ln-rYpN16j7stPziuqnla

У нас есть список релевантных пользователю товаров в обучающей выборке (то есть айдишники товаров, с которым пользователь сделал значимые интеракции на обучающем промежутке времени). И мы хотим предсказать айдишники товаров, с которыми пользователь провзаимодейтсвует на тестовом (отложенном) периоде времени. Для подсчета метрик у нас должны быть ground truth айтемы, про которые мы знаем, что они релевантны пользователю и мы их будем сравнивать с предсказанными. Ground truth айтемы в рекомендательных системах из данного датасета обычно выбираются как реальные положительные взаимодействия пользователя с айтемами в тестовом временном окне.

* Целевая переменная (таргет) — это список или множество айтемов, с которыми пользователь реально взаимодействовал в тестовом (отложенном) временном промежутке.

* Таргет формируется из положительных (значимых) взаимодействий пользователя с айтемами на отложенном периоде, например, кликов, переходов к покупке, лайков и т.п.

* Эти реальные положительные взаимодействия и есть ground truth айтемы, с которыми сравниваются предсказания модели.

* В обучающей выборке у нас есть история взаимодействий пользователя с релевантными товарами за период до теста - на её основе строится модель.

* В тесте модель должна предсказать айтемы, которые user выберет/кликнет/купит. Это и есть задача.

Иными словами, таргет это бинарная метка релевантности айтема пользователю в тестовом периоде, выраженная в виде множества ground truth айтемов. 